In [1]:
!pip install -U transformers accelerate sentencepiece protobuf nltk
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 81.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 78.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.1
    Unins

In [2]:
import torch
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import FileLink, display
from transformers import AutoProcessor

# Keep "llava" to reproduce this notebook's existing VLM.
# Other options: "qwen3vl", "clip_vit_b32", "clip_vit_b16"
VLM_CHOICE = "llava"

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Torch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


VLM_REGISTRY = {
    "qwen3vl": {
        "type": "vlm_lm",
        "model_id": "Qwen/Qwen3-VL-2B-Instruct",
        "output_prefix": "qwen3vl",
    },
    "llava": {
        "type": "vlm_lm",
        "model_id": "llava-hf/llava-1.5-7b-hf",
        "output_prefix": "llava15",
    },
    "clip_vit_b32": {
        "type": "hf_clip",
        "model_id": "openai/clip-vit-base-patch32",
        "output_prefix": "clip_vit_b32",
        "text_len": 77,
    },
    "clip_vit_b16": {
        "type": "hf_clip",
        "model_id": "openai/clip-vit-base-patch16",
        "output_prefix": "clip_vit_b16",
        "text_len": 77,
    },
}


def load_vlm(vlm_choice):
    cfg = VLM_REGISTRY[vlm_choice]
    model_id = cfg["model_id"]

    if cfg["type"] == "vlm_lm":
        if vlm_choice == "qwen3vl":
            from transformers import (
                Qwen3VLForConditionalGeneration,
            )
            model_class = Qwen3VLForConditionalGeneration

        elif vlm_choice == "llava":
            from transformers import (
                LlavaForConditionalGeneration,
            )
            model_class = LlavaForConditionalGeneration

        else:
            raise ValueError(vlm_choice)

        model = model_class.from_pretrained(
            model_id,
            torch_dtype="auto",
            device_map="auto" if torch.cuda.is_available() else None,
        )

    else:
        from transformers import CLIPModel

        dtype = (
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        )

        model = CLIPModel.from_pretrained(
            model_id,
            torch_dtype=dtype,
        ).to(device)

    processor = AutoProcessor.from_pretrained(model_id)
    tokenizer = processor.tokenizer
    model.eval()

    return model, processor, tokenizer, cfg


model, processor, tokenizer, vlm_cfg = load_vlm(VLM_CHOICE)

print("Loaded:", vlm_cfg["model_id"])
print("Model class:", type(model).__name__)

Torch: 2.10.0+cu128
Device: cuda:0
GPU: Tesla T4


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Loaded: llava-hf/llava-1.5-7b-hf
Model class: LlavaForConditionalGeneration


In [3]:
import gc
import hashlib
import json
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import FileLink, display


def find_one(filename):
    matches = list(
        Path("/kaggle/input").rglob(filename)
    )

    return matches[0]


# Alternate-class data used for fitting.
SANA_PATH = find_one(
    "geneval_augmented2000_native_exact_compact.pt"
)

# Original official GenEval pairs, used only
# after fitting to evaluate generalization.
EVAL_PAIRS_PATH = find_one(
    "sana_llava_geneval_exact_pairs.pt"
)


sana = torch.load(
    SANA_PATH,
    map_location="cpu",
    weights_only=False,
)

eval_pairs = torch.load(
    EVAL_PAIRS_PATH,
    map_location="cpu",
    weights_only=False,
)


required_keys = {
    "Y_pred_active",
    "dense_shape",
    "attention_mask",
    "input_ids",
    "special_tokens_mask",
    "offset_mapping",
    "prompt_texts",
    "conditioned_texts",
    "ids",
    "chi_prompt",
    "text_encoder_model_id",
}

missing = required_keys - set(sana)


required_eval_keys = {
    "X",
    "Y",
    "prompt_indices",
    "target_positions",
    "attention_mask",
    "target_input_ids",
    "prompt_texts",
    "conditioned_texts",
    "ids",
    "chi_prompt",
    "source_model",
    "target_model",
}

missing = required_eval_keys - set(eval_pairs)

assert not missing, (
    f"Missing official pair keys: {missing}"
)


# Everything needed is already inside the compact file.
prompts = list(sana["prompt_texts"])

conditioned_texts = list(
    sana["conditioned_texts"]
)


expected_ids = [
    hashlib.sha256(
        prompt.encode("utf-8")
    ).hexdigest()
    for prompt in prompts
]


target_mask = sana[
    "attention_mask"
].bool()

target_ids = sana["input_ids"]

target_special_mask = sana[
    "special_tokens_mask"
].bool()

target_offsets = sana[
    "offset_mapping"
]


# Reconstruct the dense fitting tensor in RAM.
target_sequences = torch.zeros(
    tuple(sana["dense_shape"]),
    dtype=sana["Y_pred_active"].dtype,
)

target_sequences[target_mask] = sana[
    "Y_pred_active"
]





print("Fitting SANA file:", SANA_PATH)
print(
    "Official evaluation pairs:",
    EVAL_PAIRS_PATH,
)
print("Fitting prompts:", len(prompts))
print(
    "Fitting target:",
    target_sequences.shape,
)
print(
    "Fitting active tokens:",
    int(target_mask.sum()),
)
print(
    "Official evaluation tokens:",
    len(eval_pairs["X"]),
)
print(
    "CHI characters:",
    len(sana["chi_prompt"]),
)

Fitting SANA file: /kaggle/input/datasets/aveenhussein/classes/geneval_augmented2000_native_exact_compact.pt
Official evaluation pairs: /kaggle/input/datasets/aveenhussein/classes/sana_llava_geneval_exact_pairs.pt
Fitting prompts: 2212
Fitting target: torch.Size([2212, 300, 2304])
Fitting active tokens: 20900
Official evaluation tokens: 4804
CHI characters: 1057


In [4]:
assert VLM_CHOICE == "llava", (
    "Set VLM_CHOICE = 'llava' in Cell 1."
)

tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
        or tokenizer.unk_token
    )

assert tokenizer.is_fast

model.eval()
model.requires_grad_(False)

# Remove tensors left behind by the failed attempt.
for variable_name in [
    "outputs",
    "source_hidden",
    "aligned",
    "source_features",
    "weights",
    "encoded",
]:
    globals().pop(variable_name, None)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


language_model = getattr(
    model,
    "language_model",
    None,
)

if language_model is None:
    language_model = model.model.language_model

source_backbone = getattr(
    language_model,
    "model",
    language_model,
)

source_input_device = (
    model.get_input_embeddings().weight.device
)

# Keep this at 1 for a 16 GB Kaggle GPU.
SOURCE_BATCH_SIZE = 1


@torch.inference_mode()
def extract_exact_pairs():
    x_parts = []
    y_parts = []
    prompt_index_parts = []
    target_position_parts = []
# 0 text with chi
    for batch_start in tqdm(
        range(
            0,
            len(conditioned_texts),
            SOURCE_BATCH_SIZE,
        ),
        desc="Embedding exact CHI + prompt inputs",
    ):
        batch_end = min(
            batch_start + SOURCE_BATCH_SIZE,
            len(conditioned_texts),
        )

        batch_texts = conditioned_texts[
            batch_start:batch_end
        ]
# 1- tokenize
        encoded = tokenizer(
            batch_texts,
            add_special_tokens=True,
            padding=True,
            truncation=False,
            return_offsets_mapping=True,
            return_special_tokens_mask=True,
            return_tensors="pt",
        )

        source_offsets = encoded.pop(
            "offset_mapping"
        )

        source_special = encoded.pop(
            "special_tokens_mask"
        ).bool()

        source_attention = encoded[
            "attention_mask"
        ].bool()

        outputs = source_backbone(
            input_ids=encoded["input_ids"].to(
                source_input_device
            ),
            attention_mask=encoded[
                "attention_mask"
            ].to(source_input_device),
            use_cache=False,
            return_dict=True,
        )

        source_hidden = getattr(
            outputs,
            "last_hidden_state",
            outputs[0],
        )

        for local_index in range(
            len(batch_texts)
        ):
            row_index = (
                batch_start + local_index
            )

            target_positions = torch.nonzero(
                target_mask[row_index],
                as_tuple=False,
            ).flatten()

            target_is_special = (
                target_special_mask[
                    row_index,
                    target_positions,
                ]
            )

            aligned = torch.empty(
                len(target_positions),
                source_hidden.shape[-1],
                device=source_hidden.device,
                dtype=torch.float32,
            )

            source_row_attention = (
                source_attention[local_index]
            )

            source_row_special = (
                source_special[local_index]
            )

            source_row_offsets = (
                source_offsets[local_index]
            )

            # Regular tokens: align using overlapping
            # character spans in CHI + prompt.
            regular_target_local = (
                torch.nonzero(
                    ~target_is_special,
                    as_tuple=False,
                ).flatten()
            )

            source_regular_indices = (
                torch.nonzero(
                    source_row_attention
                    & ~source_row_special
                    & (
                        source_row_offsets[:, 1]
                        > source_row_offsets[:, 0]
                    ),
                    as_tuple=False,
                ).flatten()
            )

            if len(regular_target_local):
                target_regular_positions = (
                    target_positions[
                        regular_target_local
                    ]
                )

                target_spans = target_offsets[
                    row_index,
                    target_regular_positions,
                ]

                source_spans = (
                    source_row_offsets[
                        source_regular_indices
                    ]
                )

                overlap_start = torch.maximum(
                    target_spans[:, None, 0],
                    source_spans[None, :, 0],
                )

                overlap_end = torch.minimum(
                    target_spans[:, None, 1],
                    source_spans[None, :, 1],
                )

                overlap = (
                    overlap_end - overlap_start
                ).clamp_min(0).float()

                missing = (
                    overlap.sum(dim=1) == 0
                )

                if missing.any():
                    target_centers = (
                        target_spans[:, 0]
                        + target_spans[:, 1]
                    ).float() / 2

                    source_centers = (
                        source_spans[:, 0]
                        + source_spans[:, 1]
                    ).float() / 2

                    missing_rows = torch.nonzero(
                        missing,
                        as_tuple=False,
                    ).flatten()

                    for missing_row in missing_rows:
                        nearest = torch.argmin(
                            torch.abs(
                                source_centers
                                - target_centers[
                                    missing_row
                                ]
                            )
                        )

                        overlap[
                            missing_row,
                            nearest,
                        ] = 1

                weights = overlap / overlap.sum(
                    dim=1,
                    keepdim=True,
                ).clamp_min(1)

                source_features = (
                    source_hidden[
                        local_index,
                        source_regular_indices.to(
                            source_hidden.device
                        ),
                    ].float()
                )

                aligned[
                    regular_target_local.to(
                        aligned.device
                    )
                ] = (
                    weights.to(aligned.device)
                    @ source_features
                )

            # Explicitly align BOS and other active
            # special tokens.
            special_target_local = (
                torch.nonzero(
                    target_is_special,
                    as_tuple=False,
                ).flatten()
            )

            source_special_indices = (
                torch.nonzero(
                    source_row_attention
                    & source_row_special,
                    as_tuple=False,
                ).flatten()
            )

            if len(special_target_local):
                assert (
                    len(source_special_indices) > 0
                ), "LLaVA produced no special token."

                for (
                    special_order,
                    target_local,
                ) in enumerate(
                    special_target_local
                ):
                    source_position = (
                        source_special_indices[
                            min(
                                special_order,
                                len(
                                    source_special_indices
                                ) - 1,
                            )
                        ]
                    )

                    aligned[
                        target_local.to(
                            aligned.device
                        )
                    ] = source_hidden[
                        local_index,
                        source_position.to(
                            source_hidden.device
                        ),
                    ].float()

            targets = target_sequences[
                row_index,
                target_positions,
            ].float()

            x_parts.append(
                aligned.to(
                    device="cpu",
                    dtype=torch.float16,
                )
            )

            y_parts.append(
                targets.cpu()
            )

            prompt_index_parts.extend(
                [row_index]
                * len(target_positions)
            )

            target_position_parts.extend(
                target_positions.tolist()
            )

        del outputs
        del source_hidden

    return (
        torch.cat(x_parts).contiguous(),
        torch.cat(y_parts).contiguous(),
        torch.tensor(
            prompt_index_parts,
            dtype=torch.int32,
        ),
        torch.tensor(
            target_position_parts,
            dtype=torch.int16,
        ),
    )


start_time = time.perf_counter()

(
    X_cpu,
    Y_cpu,
    pair_prompt_indices,
    pair_target_positions,
) = extract_exact_pairs()



PAIR_PATH = Path(
    "/kaggle/working/"
    "sana_llava_geneval_altclasses_fit_pairs.pt"
)
pair_bundle = {
    "format_version": 1,
    "kind": "sana_llava_altclasses_fit_token_pairs",
    "X": X_cpu,
    "Y": Y_cpu,
    "prompt_indices": pair_prompt_indices,
    "target_positions": pair_target_positions,
    "prompt_texts": prompts,
    "conditioned_texts": conditioned_texts,
    "ids": expected_ids,
    "attention_mask": target_mask.to(
        torch.uint8
    ),
    "target_input_ids": target_ids,
    "chi_prompt": sana["chi_prompt"],
    "source_model": vlm_cfg["model_id"],
    "target_model": sana[
        "text_encoder_model_id"
    ],
    "source_add_special_tokens": True,
    "source_tokenizer_padding_side": "right",
}

torch.save(
    pair_bundle,
    PAIR_PATH,
)

elapsed = time.perf_counter() - start_time

print("Pairs:", X_cpu.shape, Y_cpu.shape)
print("Extraction seconds:", round(elapsed, 2))
print("Saved:", PAIR_PATH)

display(FileLink(str(PAIR_PATH)))

Embedding exact CHI + prompt inputs:   0%|          | 0/2212 [00:00<?, ?it/s]

Pairs: torch.Size([20900, 4096]) torch.Size([20900, 2304])
Extraction seconds: 579.01
Saved: /kaggle/working/sana_llava_geneval_altclasses_fit_pairs.pt


/kaggle/working/sana_llava_geneval_altclasses_fit_pairs.pt

# prepare embeds

In [5]:
# -------------------------
# Prepare embeddings exactly like old calculatingw
# -------------------------

for variable_name in ["source_backbone", "language_model", "model"]:
    globals().pop(variable_name, None)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

fit_device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

X_raw = X_cpu.float().to(fit_device)
Y_raw = Y_cpu.float().to(fit_device)


def normalize_rows(T, eps=1e-12):
    return T / T.norm(
        dim=1,
        keepdim=True
    ).clamp_min(eps)


# -------------------------
# Cholesky whitening helpers
# EXACTLY the old approach
# -------------------------

def fit_cholesky_whitener(T, eps=1e-3):

    T = T.float()

    mean = T.mean(dim=0, keepdim=True)
    T_centered = T - mean

    n, d = T_centered.shape

    cov = (
        T_centered.T @ T_centered
    ) / max(n - 1, 1)

    cov = cov + eps * torch.eye(
        d,
        device=T.device,
        dtype=T.dtype
    )

    L = torch.linalg.cholesky(cov)

    T_white = torch.linalg.solve_triangular(
        L,
        T_centered.T,
        upper=False
    ).T

    state = {
        "mean": mean,
        "L": L,
    }

    return T_white, state


def apply_cholesky_whitener(T, state):

    mean = state["mean"]
    L = state["L"]

    T_centered = T - mean

    return torch.linalg.solve_triangular(
        L,
        T_centered.T,
        upper=False
    ).T


def invert_cholesky_whitener(T_white, state):

    mean = state["mean"]
    L = state["L"]

    return T_white @ L.T + mean


# ==========================================================
# X: OLD PIPELINE
# normalize -> center -> normalize
# ==========================================================

X_norm = normalize_rows(X_raw)

X_norm_mean = X_norm.mean(
    dim=0,
    keepdim=True
)

X_norm = X_norm - X_norm_mean

X_norm = normalize_rows(X_norm)


# ==========================================================
# Y: OLD PIPELINE
# normalize -> Cholesky whitening
# ==========================================================

Y_runtime = normalize_rows(Y_raw)

Y_norm, Y_chol_state = fit_cholesky_whitener(
    Y_runtime,
    eps=1e-3
)


print("Using device:", fit_device)
print("X:", X_raw.shape, "->", X_norm.shape)
print("Y:", Y_raw.shape, "->", Y_norm.shape)

Using device: cuda:0
X: torch.Size([20900, 4096]) -> torch.Size([20900, 4096])
Y: torch.Size([20900, 2304]) -> torch.Size([20900, 2304])


# Metrics

In [6]:

def print_loss(proj, target, batch_size=256):
    """
    Old calculatingw-style metrics.
    Uses batching so token-level evaluation does not allocate the full NxN matrix.
    """
    proj = normalize_rows(proj)
    target = normalize_rows(target)

    assert proj.shape == target.shape
    row_count = proj.shape[0]

    correct_values = F.cosine_similarity(proj, target, dim=1)

    wrong_sum = 0.0
    wrong_sq_sum = 0.0
    wrong_count = 0
    top1_correct = 0

    similarity_sum = 0.0
    similarity_max = -float("inf")
    similarity_min = float("inf")

    for start in range(0, row_count, batch_size):
        end = min(start + batch_size, row_count)

        S = proj[start:end] @ target.T

        local_rows = torch.arange(end - start, device=S.device)
        correct_cols = torch.arange(start, end, device=S.device)
        diag = S[local_rows, correct_cols]

        total_sum = S.sum()
        total_sq_sum = (S * S).sum()

        wrong_sum += float(total_sum - diag.sum())
        wrong_sq_sum += float(total_sq_sum - (diag * diag).sum())
        wrong_count += S.numel() - len(diag)

        top1_correct += int(
            (S.argmax(dim=1) == correct_cols).sum()
        )

        similarity_sum += float(total_sum)
        similarity_max = max(similarity_max, float(S.max()))
        similarity_min = min(similarity_min, float(S.min()))

    correct_mean = float(correct_values.mean())
    wrong_mean = wrong_sum / wrong_count
    top1 = top1_correct / row_count

    wrong_variance = max(
        wrong_sq_sum / wrong_count - wrong_mean ** 2,
        0.0,
    )
    wrong_std = wrong_variance ** 0.5

    mse = F.mse_loss(proj, target)
    l2_per_row = torch.norm(proj - target, p=2, dim=1)
    mean_l2 = float(l2_per_row.mean())
    normalized_l2_error = float((l2_per_row / 2).mean())
    l2_closeness = 1 - normalized_l2_error

    print("Correct similarity mean:", correct_mean)
    print("Wrong similarity mean:", wrong_mean)
    print("Top1 accuracy:", top1)
    print("Separation gap:", correct_mean - wrong_mean)
    print(
        "S.mean, max, min:",
        similarity_sum / (row_count * row_count),
        similarity_max,
        similarity_min,
    )
    print("correct std:", float(correct_values.std()))
    print("wrong std:", wrong_std)
    print("MSE:", float(mse))
    print("Mean L2 per row [0,2]:", mean_l2)
    print("Normalized L2 error [0,1]:", normalized_l2_error)
    print("L2 closeness [0,1]:", l2_closeness)
    print("Mean cosine similarity:", correct_mean)
    print("Min cosine similarity:", float(correct_values.min()))
    print("Max cosine similarity:", float(correct_values.max()))

    return {
        "correct_cosine": correct_mean,
        "wrong_cosine": wrong_mean,
        "top1": top1,
        "separation_gap": correct_mean - wrong_mean,
        "mse": float(mse),
        "mean_l2": mean_l2,
        "normalized_l2_error": normalized_l2_error,
        "l2_closeness": l2_closeness,
        "minimum_cosine": float(correct_values.min()),
        "maximum_cosine": float(correct_values.max()),
    }


# Krr

In [7]:
def polynomial_kernel(
    X1,
    X2,
    sigma=0.5003,
    degree=2,
    coef0=0.2
):
    return (
        sigma * (X1 @ X2.T) + coef0
    ) ** degree


sigma = 0.5003
degree = 2
coef0 = 0.2
requested_lambda = 1e-5

fit_start = time.perf_counter()


K = polynomial_kernel(
    X_norm,
    X_norm,
    sigma=sigma,
    degree=degree,
    coef0=coef0,
)

K = (K + K.T) * 0.5

I = torch.eye(
    len(K),
    device=fit_device,
    dtype=K.dtype
)


# IMPORTANT:
# fit directly to old whitened Y
rhs = Y_norm


kernel_cholesky = None
lambda_reg = None

for multiplier in [
    1, 3, 10, 30, 100, 300, 1000
]:

    candidate_lambda = (
        requested_lambda * multiplier
    )

    candidate, info = (
        torch.linalg.cholesky_ex(
            K + candidate_lambda * I
        )
    )

    if int(info.max()) == 0:
        kernel_cholesky = candidate
        lambda_reg = candidate_lambda
        break


assert kernel_cholesky is not None


# This is now the ONLY coefficient matrix.
A = torch.cholesky_solve(
    rhs,
    kernel_cholesky
)


fit_seconds = (
    time.perf_counter() - fit_start
)


print("K:", K.shape)
print("A:", A.shape)
print("Lambda:", lambda_reg)
print(
    "Fit seconds:",
    round(fit_seconds, 2)
)

K: torch.Size([20900, 20900])
A: torch.Size([20900, 2304])
Lambda: 0.0001
Fit seconds: 4.83


# project and eval

In [8]:
def transform_source(X):

    X = normalize_rows(X)

    # IMPORTANT:
    # same mean fitted on training X
    X = X - X_norm_mean

    return normalize_rows(X)


@torch.inference_mode()
def reconstruct_prediction(kernel_rows):

    # Predict in whitened Y space
    predicted_white = kernel_rows @ A

    # Undo the SAME whitening used during training
    predicted_runtime = (
        invert_cholesky_whitener(
            predicted_white,
            Y_chol_state
        )
    )

    # Old Y_runtime was row-normalized.
    return normalize_rows(
        predicted_runtime
    )


@torch.inference_mode()
def project_source_rows(
    source_rows,
    batch_size=256
):

    output_parts = []

    for start in tqdm(
        range(
            0,
            len(source_rows),
            batch_size
        ),
        desc="Projecting",
    ):

        end = min(
            start + batch_size,
            len(source_rows)
        )

        source = (
            source_rows[start:end]
            .float()
            .to(fit_device)
        )

        source = transform_source(source)

        K_test = polynomial_kernel(
            source,
            X_norm,
            sigma=sigma,
            degree=degree,
            coef0=coef0,
        )

        output_parts.append(
            reconstruct_prediction(
                K_test
            ).cpu()
        )

    return torch.cat(
        output_parts
    ).contiguous()


# ==========================================
# TRAIN
# ==========================================

Y_pred = reconstruct_prediction(K)

print(
    "TRAIN / ALTERNATE-CLASS PAIRS"
)

train_metrics = print_loss(
    Y_pred,
    Y_runtime
)


# ==========================================
# OFFICIAL GENEVAL / HELD OUT
# ==========================================

Y_pred_eval_cpu = project_source_rows(
    eval_pairs["X"]
)

Y_eval_cpu = (
    eval_pairs["Y"].float()
)

# Old pipeline compares against
# row-normalized runtime Y.
Y_eval_runtime = normalize_rows(
    Y_eval_cpu.to(fit_device)
)


print(
    "\nOFFICIAL GENEVAL / HELD-OUT PAIRS"
)

eval_metrics = print_loss(
    Y_pred_eval_cpu.to(fit_device),
    Y_eval_runtime,
)

TRAIN / ALTERNATE-CLASS PAIRS
Correct similarity mean: 0.9997789263725281
Wrong similarity mean: 0.4255872943029027
Top1 accuracy: 0.3543540669856459
Separation gap: 0.5741916320696254
S.mean, max, min: 0.42561477358576955 0.999997615814209 -0.36331361532211304
correct std: 0.005133865866810083
wrong std: 0.3373350260193967
MSE: 1.919252952120587e-07
Mean L2 per row [0,2]: 0.013301205821335316
Normalized L2 error [0,1]: 0.006650602910667658
L2 closeness [0,1]: 0.9933493970893323
Mean cosine similarity: 0.9997789263725281
Min cosine similarity: 0.31878453493118286
Max cosine similarity: 0.9999970197677612


Projecting:   0%|          | 0/19 [00:00<?, ?it/s]


OFFICIAL GENEVAL / HELD-OUT PAIRS
Correct similarity mean: 0.9739966988563538
Wrong similarity mean: 0.46795629163955776
Top1 accuracy: 0.197751873438801
Separation gap: 0.506040407216796
S.mean, max, min: 0.46806162682482194 0.9999997019767761 -0.2876037359237671
correct std: 0.050742100924253464
wrong std: 0.3401214038247455
MSE: 2.25724379561143e-05
Mean L2 per row [0,2]: 0.13622325658798218
Normalized L2 error [0,1]: 0.06811162829399109
L2 closeness [0,1]: 0.9318883717060089
Mean cosine similarity: 0.9739966988563538
Min cosine similarity: 0.19220253825187683
Max cosine similarity: 0.9999992847442627


In [9]:
def print_norms(name, tensor):
    norms = tensor.float().norm(dim=-1).flatten()
    print(
        f"{name:24s}",
        f"min={norms.min().item():.3f}",
        f"median={norms.median().item():.3f}",
        f"max={norms.max().item():.3f}",
    )


print_norms("Native targets", Y_raw)
print_norms("Predicted direction", predicted_direction)
print_norms("Predicted raw/scaled", predicted_raw)

print(
    "Predicted scalar norms:",
    f"min={predicted_norm.min().item():.3f}",
    f"median={predicted_norm.median().item():.3f}",
    f"max={predicted_norm.max().item():.3f}",
)

assert predicted_direction.norm(dim=1).median() < 2
assert predicted_raw.norm(dim=1).median() > 10

Native targets           min=90.439 median=117.187 max=260.020


NameError: name 'predicted_direction' is not defined

# Save projector and projected official GenEval embeddings

In [66]:

PROJECTOR_PATH = Path(
    "/kaggle/working/sana_llava_geneval_altclasses_projector.pt"
)
PROJECTED_PATH = Path(
    "/kaggle/working/sana_llava_geneval_altclasses_projected_compact1.pt"
)
projector_bundle = {
    "format_version": 1,
    "kind": "sana_altclasses_token_kernel_projector",
    "source_model": vlm_cfg["model_id"],
    "target_model": sana["text_encoder_model_id"],
    "source_representation": "contextual_hidden_states_of_exact_chi_plus_prompt",
    "target_representation": "native_sana_gemma_sequence",
    "source_add_special_tokens": True,
    "source_tokenizer_padding_side": "right",

    "prompt_texts": prompts,
    "ids": expected_ids,
    "chi_prompt": sana["chi_prompt"],
    "prompt_indices": pair_prompt_indices,
    "target_positions": pair_target_positions,

    # X preprocessing
    "X_norm_mean": X_norm_mean.cpu(),
    "train_features": X_norm.cpu(),

    "kernel": {
        "kind": "polynomial",
        "sigma": sigma,
        "degree": degree,
        "coef0": coef0,
        "lambda_reg": lambda_reg,
    },

    # KRR coefficients
    "coefficients": A.cpu(),

    # Y whitening
    "Y_mean": Y_chol_state["mean"].cpu(),
    "Y_cholesky": Y_chol_state["L"].cpu(),
    "Y_cholesky_epsilon": 1e-3,

    "training_metrics": train_metrics,
    "heldout_metrics": eval_metrics,
    "fit_seconds": fit_seconds,
    "official_geneval_used_for_fitting": False,
}

torch.save(projector_bundle, PROJECTOR_PATH)

eval_mask = eval_pairs["attention_mask"].bool()
mask_coordinates = torch.nonzero(eval_mask, as_tuple=False)

assert torch.equal(
    mask_coordinates[:, 0].to(torch.int32),
    eval_pairs["prompt_indices"],
)
assert torch.equal(
    mask_coordinates[:, 1].to(torch.int16),
    eval_pairs["target_positions"],
)
assert len(Y_pred_eval_cpu) == int(eval_mask.sum())

projected_bundle = {
    "format_version": 2,
    "kind": "sana_projected_official_geneval_native_sequence",
    "projected_text_mode": "native_sequence",
    "Y_pred_active": Y_pred_eval_cpu.to(torch.bfloat16),
    "conditioning_storage": "active_tokens_only",
    "dense_shape": [553, 300, 2304],
    "attention_mask": eval_pairs["attention_mask"].clone(),
    "token_ids": eval_pairs["target_input_ids"].clone(),
    "prompt_texts": list(eval_pairs["prompt_texts"]),
    "conditioned_texts": list(eval_pairs["conditioned_texts"]),
    "ids": list(eval_pairs["ids"]),
    "chi_prompt": eval_pairs["chi_prompt"],
    "source_model": eval_pairs["source_model"],
    "target_model": eval_pairs["target_model"],
    "conditioning_layout": "exact_native_sana_positions",
    "projector_kind": projector_bundle["kind"],
    "projector_training_metrics": train_metrics,
    "projector_heldout_metrics": eval_metrics,
}

torch.save(projected_bundle, PROJECTED_PATH)

print("Projector:", PROJECTOR_PATH)
print("Projected eval file:", PROJECTED_PATH)
print("Projected active shape:", Y_pred_eval_cpu.shape)

display(FileLink(str(PROJECTOR_PATH)))
display(FileLink(str(PROJECTED_PATH)))


Projector: /kaggle/working/sana_llava_geneval_altclasses_projector.pt
Projected eval file: /kaggle/working/sana_llava_geneval_altclasses_projected_compact1.pt
Projected active shape: torch.Size([4804, 2304])


/kaggle/working/sana_llava_geneval_altclasses_projector.pt

/kaggle/working/sana_llava_geneval_altclasses_projected_compact1.pt

In [71]:
import os
from pathlib import Path
from IPython.display import FileLink, display

# Ensure this really is the new 2,000-class calculation run.
assert len(prompts) == 2212, (
    f"Expected 2,212 fitting prompts, got {len(prompts)}"
)

projector_bundle["kind"] = (
    "sana_augmented2000_token_kernel_projector"
)

projector_bundle["calculation_prompt_count"] = len(prompts)

projected_bundle["projector_kind"] = (
    projector_bundle["kind"]
)

projected_bundle["projector_training_prompt_count"] = len(prompts)

PROJECTOR_PATH = Path(
    "/kaggle/working/"
    "sana_llava_geneval_augmented2000_projector.pt"
)

PROJECTED_PATH = Path(
    "/kaggle/working/"
    "sana_llava_geneval_augmented2000_projected_compact.pt"
)

torch.save(projector_bundle, PROJECTOR_PATH)
torch.save(projected_bundle, PROJECTED_PATH)

print("Saved:", PROJECTED_PATH)
print(
    "Size:",
    f"{PROJECTED_PATH.stat().st_size / 1024**2:.1f} MB",
)

# Relative link prevents the 404.
os.chdir("/kaggle/working")
display(FileLink(PROJECTED_PATH.name))

Saved: /kaggle/working/sana_llava_geneval_augmented2000_projected_compact.pt
Size: 22.5 MB


/kaggle/working/sana_llava_geneval_augmented2000_projected_compact.pt

In [49]:
conditioned_texts[1]

'Given a user prompt, generate an "Enhanced prompt" that provides detailed visual descriptions suitable for image generation. Evaluate the level of detail in the user prompt:\n- If the prompt is simple, focus on adding specifics about colors, shapes, sizes, textures, and spatial relationships to create vivid and concrete scenes.\n- If the prompt is already detailed, refine and enhance the existing details slightly without overcomplicating.\nHere are examples of how to transform or refine prompts:\n- User Prompt: A cat sleeping -> Enhanced: A small, fluffy white cat curled up in a round shape, sleeping peacefully on a warm sunny windowsill, surrounded by pots of blooming red flowers.\n- User Prompt: A busy city street -> Enhanced: A bustling city street scene at dusk, featuring glowing street lamps, a diverse crowd of people in colorful clothing, and a double-decker bus passing by towering glass skyscrapers.\nPlease generate only the enhanced description for the prompt below and avoid i

In [50]:
# ============================================================
# ORIGINAL OFFICIAL PAIRS: IN-SAMPLE SANITY CHECK
#
# Fits and evaluates on the same official token pairs.
# This is the setup that should produce near-perfect cosine.
# It does NOT modify the alternate-class projector.
# ============================================================

base_X_raw = (
    eval_pairs["X"]
    .float()
    .to(fit_device)
)

base_Y_raw = (
    eval_pairs["Y"]
    .float()
    .to(fit_device)
)


# Same source preprocessing.
base_X_norm = normalize_rows(base_X_raw)

base_X_mean = base_X_norm.mean(
    dim=0,
    keepdim=True,
)

base_X_norm = normalize_rows(
    base_X_norm - base_X_mean
)


# Same target preprocessing.
base_Y_runtime = normalize_rows(
    base_Y_raw
)

(
    base_Y_white,
    base_Y_chol_state,
) = fit_cholesky_whitener(
    base_Y_runtime,
    eps=1e-3,
)


# Fit kernel on the original official pairs.
base_K = polynomial_kernel(
    base_X_norm,
    base_X_norm,
    sigma=sigma,
    degree=degree,
    coef0=coef0,
)

base_K = (
    base_K + base_K.T
) * 0.5

base_identity = torch.eye(
    len(base_K),
    device=fit_device,
    dtype=base_K.dtype,
)

base_kernel_cholesky = None
base_lambda = None

for multiplier in [
    1,
    3,
    10,
    30,
    100,
    300,
    1000,
]:
    candidate_lambda = (
        requested_lambda * multiplier
    )

    candidate, info = (
        torch.linalg.cholesky_ex(
            base_K
            + candidate_lambda
            * base_identity
        )
    )

    if int(info.max()) == 0:
        base_kernel_cholesky = candidate
        base_lambda = candidate_lambda
        break

assert base_kernel_cholesky is not None


base_coefficients = torch.cholesky_solve(
    base_Y_white,
    base_kernel_cholesky,
)


# Project the same official pairs used for fitting.
base_predicted_white = (
    base_K @ base_coefficients
)

base_predicted = normalize_rows(
    invert_cholesky_whitener(
        base_predicted_white,
        base_Y_chol_state,
    )
)


print()
print("ORIGINAL OFFICIAL PAIRS / IN-SAMPLE")
print("===================================")
print("Rows:", len(base_X_raw))
print("Kernel:", tuple(base_K.shape))
print("Lambda:", base_lambda)

base_in_sample_metrics = print_loss(
    base_predicted,
    base_Y_runtime,
)

print(
    "Raw normalized MSE:",
    float(
        F.mse_loss(
            base_predicted,
            base_Y_runtime,
        )
        / base_Y_runtime.pow(2)
        .mean()
        .clamp_min(1e-12)
    ),
)


# Release the additional comparison kernel.
del base_K
del base_identity
del base_kernel_cholesky
del base_coefficients
del base_predicted_white

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


ORIGINAL OFFICIAL PAIRS / IN-SAMPLE
Rows: 4804
Kernel: (4804, 4804)
Lambda: 1e-05
Correct similarity mean: 1.0
Wrong similarity mean: 0.4578846362459419
Top1 accuracy: 0.2995420482930891
Separation gap: 0.5421153637540581
S.mean, max, min: 0.4579974829078391 1.0000005960464478 -0.2875148057937622
correct std: 8.167801723857337e-08
wrong std: 0.3375734567749167
MSE: 1.864399780393544e-11
Mean L2 per row [0,2]: 0.00017783768998924643
Normalized L2 error [0,1]: 8.891884499462321e-05
L2 closeness [0,1]: 0.9999110811550054
Mean cosine similarity: 1.0
Min cosine similarity: 0.9999997615814209
Max cosine similarity: 1.000000238418579
Raw normalized MSE: 4.295578293067592e-08


In [51]:
# ============================================================
# CLASS-TOKEN-ONLY METRICS
#
# BASE = original official-pair projector
# NEW  = augmented-class projector
#
# Evaluation remains on the 80 held-out official classes.
# ============================================================

import re
import torch
from transformers import AutoTokenizer


required_variables = [
    "eval_pairs",
    "base_predicted",
    "Y_pred_eval_cpu",
    "Y_eval_runtime",
    "fit_device",
    "print_loss",
]

missing = [
    name
    for name in required_variables
    if name not in globals()
]

assert not missing, (
    "Run through the base and augmented projector cells first. "
    "Missing: " + ", ".join(missing)
)


OFFICIAL_CLASSES = """
person
bicycle
car
motorcycle
airplane
bus
train
truck
boat
traffic light
fire hydrant
stop sign
parking meter
bench
bird
cat
dog
horse
sheep
cow
elephant
bear
zebra
giraffe
backpack
umbrella
handbag
tie
suitcase
frisbee
skis
snowboard
sports ball
kite
baseball bat
baseball glove
skateboard
surfboard
tennis racket
bottle
wine glass
cup
fork
knife
spoon
bowl
banana
apple
sandwich
orange
broccoli
carrot
hot dog
pizza
donut
cake
chair
couch
potted plant
bed
dining table
toilet
tv
laptop
computer mouse
tv remote
computer keyboard
cell phone
microwave
oven
toaster
sink
refrigerator
book
clock
vase
scissors
teddy bear
hair drier
toothbrush
""".strip().splitlines()

assert len(OFFICIAL_CLASSES) == 80
assert len(set(OFFICIAL_CLASSES)) == 80


RELATIONS = [
    " left of ",
    " right of ",
    " above ",
    " below ",
]

COLOR_WORDS = {
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "purple",
    "pink",
    "brown",
    "black",
    "white",
}


def find_object_spans(prompt):
    candidates = []

    for class_name in OFFICIAL_CLASSES:
        pattern = (
            r"(?<!\w)"
            + re.escape(class_name)
            + r"(?:es|s)?(?!\w)"
        )

        for match in re.finditer(pattern, prompt):
            candidates.append({
                "start": match.start(),
                "end": match.end(),
                "class_name": class_name,
            })

    # Remove shorter matches inside longer classes.
    candidates = [
        candidate
        for candidate in candidates
        if not any(
            other["start"] <= candidate["start"]
            and candidate["end"] <= other["end"]
            and (
                other["end"] - other["start"]
                > candidate["end"] - candidate["start"]
            )
            for other in candidates
        )
    ]

    candidates.sort(
        key=lambda item: item["start"]
    )

    expected_count = (
        2
        if (
            " and " in prompt
            or any(
                relation in prompt
                for relation in RELATIONS
            )
        )
        else 1
    )

    # "orange" may be a color rather than the object.
    while len(candidates) > expected_count:
        removed = False

        for index, candidate in enumerate(candidates):
            if candidate["class_name"] not in COLOR_WORDS:
                continue

            boundaries = [
                prompt.find(
                    separator,
                    candidate["end"],
                )
                for separator in [
                    " and ",
                    *RELATIONS,
                ]
            ]

            boundaries = [
                position
                for position in boundaries
                if position >= 0
            ]

            phrase_end = (
                min(boundaries)
                if boundaries
                else len(prompt)
            )

            has_later_object = any(
                other["start"] >= candidate["end"]
                and other["start"] < phrase_end
                for other in candidates
            )

            if has_later_object:
                candidates.pop(index)
                removed = True
                break

        if not removed:
            break

    assert len(candidates) == expected_count, (
        f"Could not identify objects in {prompt!r}\n"
        f"Candidates: {candidates}"
    )

    return candidates


# ============================================================
# Reconstruct exact Gemma token offsets.
# ============================================================

target_tokenizer = AutoTokenizer.from_pretrained(
    eval_pairs["target_model"],
    use_fast=True,
)

assert target_tokenizer.is_fast

target_tokenizer.padding_side = "right"

if target_tokenizer.pad_token_id is None:
    target_tokenizer.pad_token = (
        target_tokenizer.eos_token
        or target_tokenizer.unk_token
    )


sequence_length = int(
    eval_pairs["attention_mask"].shape[1]
)

chi_prompt = eval_pairs["chi_prompt"]

tokenizer_max_length = (
    len(target_tokenizer.encode(chi_prompt))
    + sequence_length
    - 2
)

target_encoded = target_tokenizer(
    list(eval_pairs["conditioned_texts"]),
    max_length=tokenizer_max_length,
    padding="max_length",
    truncation=True,
    return_offsets_mapping=True,
    return_tensors="pt",
)

select_index = torch.cat([
    torch.zeros(
        1,
        dtype=torch.long,
    ),
    torch.arange(
        tokenizer_max_length
        - sequence_length
        + 1,
        tokenizer_max_length,
    ),
])

selected_offsets = target_encoded[
    "offset_mapping"
].index_select(
    1,
    select_index,
)

selected_input_ids = target_encoded[
    "input_ids"
].index_select(
    1,
    select_index,
).to(torch.int32)


assert torch.equal(
    selected_input_ids,
    eval_pairs["target_input_ids"],
), "Reconstructed Gemma tokens do not match eval_pairs"


pair_prompt_indices_cpu = (
    eval_pairs["prompt_indices"]
    .long()
    .cpu()
)

pair_target_positions_cpu = (
    eval_pairs["target_positions"]
    .long()
    .cpu()
)

pair_offsets = selected_offsets[
    pair_prompt_indices_cpu,
    pair_target_positions_cpu,
]


# ============================================================
# Locate every official object-class occurrence.
# ============================================================

class_row_groups = []
class_occurrence_names = []

prompt_offset = len(chi_prompt)

for prompt_index, prompt in enumerate(
    eval_pairs["prompt_texts"]
):
    for object_span in find_object_spans(prompt):
        absolute_start = (
            prompt_offset
            + object_span["start"]
        )

        absolute_end = (
            prompt_offset
            + object_span["end"]
        )

        rows = torch.nonzero(
            (
                pair_prompt_indices_cpu
                == prompt_index
            )
            & (
                pair_offsets[:, 1]
                > absolute_start
            )
            & (
                pair_offsets[:, 0]
                < absolute_end
            )
            & (
                pair_offsets[:, 1]
                > pair_offsets[:, 0]
            ),
            as_tuple=False,
        ).flatten()

        assert len(rows) > 0, (
            f"No Gemma token overlaps "
            f"{object_span['class_name']!r} "
            f"in {prompt!r}"
        )

        class_row_groups.append(rows)
        class_occurrence_names.append(
            object_span["class_name"]
        )


assert len(class_row_groups) == 852

print("Object occurrences:", len(class_row_groups))
print(
    "Unique official classes:",
    len(set(class_occurrence_names)),
)


def pool_object_occurrences(tensor):
    pooled = []

    for rows in class_row_groups:
        rows = rows.to(tensor.device)

        pooled.append(
            tensor.index_select(
                0,
                rows,
            ).mean(dim=0)
        )

    return torch.stack(pooled)


def average_by_class(occurrence_tensor):
    class_averages = []

    for class_name in OFFICIAL_CLASSES:
        occurrence_indices = [
            index
            for index, name in enumerate(
                class_occurrence_names
            )
            if name == class_name
        ]

        assert occurrence_indices, (
            f"No occurrence found for {class_name}"
        )

        indices = torch.tensor(
            occurrence_indices,
            device=occurrence_tensor.device,
            dtype=torch.long,
        )

        class_averages.append(
            occurrence_tensor.index_select(
                0,
                indices,
            ).mean(dim=0)
        )

    return torch.stack(class_averages)


# ============================================================
# Pool native target, base output, and augmented output.
# ============================================================

target_occurrences = pool_object_occurrences(
    Y_eval_runtime
)

base_occurrences = pool_object_occurrences(
    base_predicted
)

augmented_occurrences = pool_object_occurrences(
    Y_pred_eval_cpu.to(fit_device)
)


target_class_averages = average_by_class(
    target_occurrences
)

base_class_averages = average_by_class(
    base_occurrences
)

augmented_class_averages = average_by_class(
    augmented_occurrences
)


# ============================================================
# Metrics over the 80 averaged held-out classes.
# ============================================================

print()
print("BASE PROJECTOR: 80 CLASS AVERAGES")
print("=================================")

base_class_metrics = print_loss(
    base_class_averages,
    target_class_averages,
)


print()
print("AUGMENTED PROJECTOR: 80 CLASS AVERAGES")
print("======================================")

augmented_class_metrics = print_loss(
    augmented_class_averages,
    target_class_averages,
)


print()
print("SUMMARY")
print("=======")
print(
    f"{'Metric':<24}"
    f"{'Base':>14}"
    f"{'Augmented':>14}"
)

for metric_name in [
    "correct_cosine",
    "wrong_cosine",
    "separation_gap",
    "top1",
    "mse",
    "mean_l2",
    "l2_closeness",
    "minimum_cosine",
    "maximum_cosine",
]:
    print(
        f"{metric_name:<24}"
        f"{base_class_metrics[metric_name]:>14.6f}"
        f"{augmented_class_metrics[metric_name]:>14.6f}"
    )

Object occurrences: 852
Unique official classes: 80

BASE PROJECTOR: 80 CLASS AVERAGES
Correct similarity mean: 1.0
Wrong similarity mean: 0.7783091871044304
Top1 accuracy: 1.0
Separation gap: 0.22169081289556958
S.mean, max, min: 0.781080322265625 1.0000001192092896 0.4744776487350464
correct std: 5.8908970146376305e-08
wrong std: 0.06777435924387229
MSE: 7.927692356746174e-12
Mean L2 per row [0,2]: 0.00012972937838640064
Normalized L2 error [0,1]: 6.486468919320032e-05
L2 closeness [0,1]: 0.9999351353108068
Mean cosine similarity: 1.0
Min cosine similarity: 0.9999998807907104
Max cosine similarity: 1.000000238418579

AUGMENTED PROJECTOR: 80 CLASS AVERAGES
Correct similarity mean: 0.9456753134727478
Wrong similarity mean: 0.8083781182011471
Top1 accuracy: 0.8875
Separation gap: 0.13729719527160067
S.mean, max, min: 0.8100942993164062 0.9979745745658875 0.524800181388855
correct std: 0.03965189680457115
wrong std: 0.05825984002700698
MSE: 4.715690738521516e-05
Mean L2 per row [0,2]: 0.